# Entraînement du modèle « type de défaut » (NEU-DET)

Ce modèle reconnaît 6 défauts de surface de l'acier (jeu NEU-DET). Dans l'application, il précise le **type** de
dommage une fois que le modèle v2 a jugé la pièce endommagée.

Données attendues : `data/images/NEU-DET/train/images/<classe>/` et `data/images/NEU-DET/validation/images/<classe>/`.
- `train/` : 85 % entraînement, 15 % validation (choix du meilleur modèle);
- `validation/` : **test**, jamais utilisé pendant l'entraînement.

Cliquez sur **Run All** (environ 10 minutes).

In [ ]:
import json
import sys
import time
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from diagnostic.vision import ECART_TYPE, MODELE_NEU, MOYENNE, creer_resnet, transfo_neu

DOSSIER_NEU = RACINE / "data" / "images" / "NEU-DET"
DOSSIER_RESULTATS = RACINE / "resultats"
GRAINE, EPOQUES, PART_VALIDATION = 42, 8, 0.15
torch.manual_seed(GRAINE)
plt.rcParams["figure.dpi"] = 110

## 1. Données

L'augmentation (retournements, rotations, luminosité) ne sert qu'à l'entraînement. La validation et le test utilisent
`transfo_neu`, la même transformation que l'application (`diagnostic/vision.py`).

In [ ]:
transfo_entrainement = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MOYENNE, ECART_TYPE),
])

base_entr = datasets.ImageFolder(DOSSIER_NEU / "train" / "images", transform=transfo_entrainement)
base_val = datasets.ImageFolder(DOSSIER_NEU / "train" / "images", transform=transfo_neu)
test = datasets.ImageFolder(DOSSIER_NEU / "validation" / "images", transform=transfo_neu)
CLASSES = base_entr.classes
assert CLASSES == test.classes, "Classes différentes entre train et test"

# Découpage stratifié : 15 % de chaque classe pour la validation
generateur = torch.Generator().manual_seed(GRAINE)
idx_entr, idx_val = [], []
for classe in range(len(CLASSES)):
    indices = [i for i, cible in enumerate(base_entr.targets) if cible == classe]
    melange = torch.randperm(len(indices), generator=generateur).tolist()
    n_val = round(len(indices) * PART_VALIDATION)
    idx_val += [indices[j] for j in melange[:n_val]]
    idx_entr += [indices[j] for j in melange[n_val:]]

chargeurs = {
    "train": DataLoader(Subset(base_entr, idx_entr), batch_size=32, shuffle=True, num_workers=0),
    "val": DataLoader(Subset(base_val, idx_val), batch_size=32, num_workers=0),
    "test": DataLoader(test, batch_size=32, num_workers=0),
}
pd.DataFrame({"Images": {nom: len(ch.dataset) for nom, ch in chargeurs.items()}}).T

## 2. Entraînement

In [ ]:
modele = creer_resnet(len(CLASSES))
optimiseur = torch.optim.AdamW(modele.parameters(), lr=3e-4, weight_decay=1e-4)
planificateur = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=EPOQUES)
perte_fn = nn.CrossEntropyLoss()

def exactitude(chargeur):
    modele.eval()
    corrects = total = 0
    with torch.no_grad():
        for x, y in chargeur:
            corrects += (modele(x).argmax(1) == y).sum().item()
            total += len(y)
    return corrects / total

historique, meilleure = [], 0.0
MODELE_NEU.parent.mkdir(exist_ok=True)
for epoque in range(1, EPOQUES + 1):
    debut = time.time()
    modele.train()
    perte_totale = 0.0
    for i, (x, y) in enumerate(chargeurs["train"], 1):
        optimiseur.zero_grad()
        perte = perte_fn(modele(x), y)
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item()
        print(f"\r  Époque {epoque}/{EPOQUES}  lot {i}/{len(chargeurs['train'])}", end="")
    planificateur.step()
    exact_val = exactitude(chargeurs["val"])
    historique.append({"epoque": epoque, "perte": perte_totale / len(chargeurs["train"]), "exactitude_validation": exact_val})
    print(f"\r  Époque {epoque}/{EPOQUES}  perte {historique[-1]['perte']:.3f}  "
          f"exactitude validation {exact_val:.1%}  ({time.time() - debut:.0f} s)")
    if exact_val >= meilleure:
        meilleure = exact_val
        torch.save({"etat": modele.state_dict(), "classes": CLASSES, "epoque": epoque}, MODELE_NEU)

print(f"\nMeilleure exactitude en validation : {meilleure:.1%}  →  modèle enregistré dans {MODELE_NEU.name}")
DOSSIER_RESULTATS.mkdir(exist_ok=True)
_ = (DOSSIER_RESULTATS / "vision_historique.json").write_text(json.dumps(historique, indent=2), encoding="utf-8")

## 3. Évaluation sur le jeu de test

Critères visés : rappel moyen ≥ 90 % et précision moyenne ≥ 85 %.

In [ ]:
sauvegarde = torch.load(MODELE_NEU, map_location="cpu", weights_only=True)
modele.load_state_dict(sauvegarde["etat"])
modele.eval()

vraies, predites = [], []
with torch.no_grad():
    for x, y in chargeurs["test"]:
        predites += modele(x).argmax(1).tolist()
        vraies += y.tolist()

rapport = classification_report(vraies, predites, target_names=CLASSES, output_dict=True, zero_division=0)
rappel, precision = rapport["macro avg"]["recall"], rapport["macro avg"]["precision"]
print(f"Modèle de l'époque {sauvegarde['epoque']}")
print(f"Rappel moyen      : {rappel:.1%}  (critère ≥ 90 %) → {'ATTEINT' if rappel >= 0.90 else 'NON ATTEINT'}")
print(f"Précision moyenne : {precision:.1%}  (critère ≥ 85 %) → {'ATTEINT' if precision >= 0.85 else 'NON ATTEINT'}")
_ = (DOSSIER_RESULTATS / "vision_metriques.json").write_text(json.dumps(rapport, ensure_ascii=False, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(vraies, predites), display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set(title="Matrice de confusion – jeu de test NEU-DET", xlabel="Classe prédite", ylabel="Classe réelle")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(DOSSIER_RESULTATS / "vision_matrice_confusion.png", dpi=150)
plt.show()
pd.DataFrame(rapport).T.iloc[:len(CLASSES)].style.format("{:.3f}", subset=["precision", "recall", "f1-score"])